# Topic 16 — Feature Selection

## Keeping the features that help, dropping the ones that don't — filter, wrapper and embedded methods

> **Analogy:** Packing for a trip with a small suitcase. You can't take everything, so you
> (1) throw out obvious junk (empty bottles = **constant** columns, two identical shirts = **duplicates**),
> (2) quickly rate each item on its own (**filter** methods),
> (3) actually try packing different combinations and see which fits best (**wrapper** methods), or
> (4) let a smart suitcase that compresses unused space decide while you pack (**embedded** methods).

### Roadmap

| Part | Sections |
|---|---|
| 🟢 **Foundations** | 1 What & why · 2 Selection vs extraction · 3 The three families · 4 Dataset & baseline |
| 🟡 **Filter methods** | 5 Constant & quasi-constant · 6 Duplicate & highly correlated · 7 Univariate tests (ANOVA F, chi², mutual information) · 8 `SelectKBest` / `SelectPercentile` |
| 🟡 **Wrapper methods** | 9 RFE & RFECV · 10 Sequential feature selection |
| 🔴 **Embedded & model-agnostic** | 11 L1 (Lasso) · 12 Tree importance · 13 Permutation importance |
| 🔴 **Doing it right** | 14 Selection leakage · 15 Tuning *k* inside a pipeline · 16 Stability · 17 Categorical features · 18 Choosing a method · 19 Common mistakes |
| 📝 **Practice** | 18 problems → Solutions |
| 🚀 **Project** | Feature selection for red-wine quality |
| 📋 **Reference** | Cheat sheet |

**Builds on:** EDA (Topic 7), encoding (Topic 6), scaling (Topic 4), correlation & VIF (Topic 12), statistics (Topic 14).
Model training and cross-validation are covered fully in **Topic 17** — here we use them as tools.

> 📚 **Course navigation** — you are in **Topic 16: Feature Selection**
> ⬅️ Previous: Topic 15 — Probability for Data Science & ML  ·  ➡️ Next: Topic 17 — Machine Learning

<details><summary>Full course sequence (click to expand)</summary>

| # | Topic | Notebook |
|---|---|---|
| 01 | Python Basics | `Topic_01_Python_Basics.ipynb` |
| 02 | Pandas DataFrame Basics | `Topic_02_Pandas_DataFrame_Basics.ipynb` |
| 03 | Handling Missing Values | `Topic_03_Handling_Missing_Values.ipynb` |
| 04 | Normalization & Standardization | `Topic_04_Normalization_Standardization.ipynb` |
| 05 | Detecting & Treating Outliers | `Topic_05_Outliers.ipynb` |
| 06 | Encoding Techniques | `Topic_06_Encoding_Techniques.ipynb` |
| 07 | Exploratory Data Analysis (EDA) | `Topic_07_Exploratory_Data_Analysis.ipynb` |
| 08 | Data Visualization | `Topic_08_Data_Visualization.ipynb` |
| 09 | Pandas Data Manipulation | `Topic_09_Pandas_Data_Manipulation.ipynb` |
| 10 | Combining DataFrames | `Topic_10_Combining_DataFrames.ipynb` |
| 11 | Feature Engineering | `Topic_11_Feature_Engineering.ipynb` |
| 11b | KBinsDiscretizer (Discretization) | `Topic_11b_KBinsDiscretizer.ipynb` |
| 12 | Correlation & Multicollinearity | `Topic_12_Correlation_Multicollinearity.ipynb` |
| 13 | Data Transformation & Skewness | `Topic_13_Data_Transformation_Skewness.ipynb` |
| 14 | Statistics for Data Science | `Topic_14_Statistics_for_Data_Science.ipynb` |
| 15 | Probability for Data Science & ML | `Topic_15_Probability_for_Data_Science_ML.ipynb` |
| **▶ 16 ◀** | **Feature Selection** | `Topic_16_Feature_Selection.ipynb` |
| 17 | Machine Learning | `Topic_17_Machine_Learning.ipynb` |

</details>

## 📋 Lesson Index — What This Lesson Covers

This lesson has **21 sections**. Click any one to jump to it.

- [1. What Is Feature Selection and Why Do It?](#sec-1)
- [2. Feature Selection vs Feature Extraction](#sec-2)
- [3. The Three Families of Methods](#sec-3)
- [4. The Dataset and a Baseline](#sec-4)
- [5. Filter — Constant and Quasi-Constant Features](#sec-5)
- [6. Filter — Duplicate and Highly Correlated Features](#sec-6)
- [7. Filter — Univariate Statistical Tests](#sec-7)
- [8. `SelectKBest` and `SelectPercentile`](#sec-8)
- [9. Wrapper — Recursive Feature Elimination (RFE / RFECV)](#sec-9)
- [10. Wrapper — Sequential Feature Selection](#sec-10)
- [11. Embedded — L1 Regularisation (Lasso)](#sec-11)
- [12. Embedded — Tree-Based Importance](#sec-12)
- [13. Permutation Importance (model-agnostic)](#sec-13)
- [14. ⚠️ Selection Leakage — the Most Important Section](#sec-14)
- [15. Tuning the Number of Features with a Pipeline](#sec-15)
- [16. Stability — Would You Pick the Same Features Again?](#sec-16)
- [17. Categorical Features](#sec-17)
- [18. Comparing Methods and Choosing One](#sec-18)
- [19. Common Mistakes](#sec-19)
- [🚀 Project — Feature Selection for Red-Wine Quality](#sec-20)
- [📋 Feature Selection Cheat Sheet](#sec-21)

> 🗂️ Full course list: [Topic_00_All_Topics_Index.ipynb](Topic_00_All_Topics_Index.ipynb)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression, LassoCV, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_selection import (VarianceThreshold, SelectKBest, SelectPercentile, f_classif, chi2,
                                       mutual_info_classif, f_regression, mutual_info_regression,
                                       RFE, RFECV, SequentialFeatureSelector, SelectFromModel)
from sklearn.inspection import permutation_importance

pd.set_option('display.width', 130)
pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')
import sklearn; print('scikit-learn', sklearn.__version__)

---
<a id="sec-1"></a>
# 1. What Is Feature Selection and Why Do It?

### What
Choosing a **subset of the existing columns** to use as model inputs, and discarding the rest.

### Why
| Benefit | Explanation |
|---|---|
| **Less overfitting** | every useless column is a chance for the model to learn noise |
| **Better generalisation** | especially when rows are few and columns many (the *curse of dimensionality*) |
| **Faster training & prediction** | fewer columns to compute and store |
| **Simpler, explainable models** | "these 8 measurements matter" is easier to trust than 300 |
| **Cheaper data collection** | in production you only need to collect the selected features |

### When
- many columns relative to rows (e.g. 200 features, 500 rows);
- after one-hot encoding exploded the column count;
- obvious junk exists (constant, duplicate, ID-like columns);
- you need an interpretable model or a cheaper data pipeline.

### When *not*
- few features and plenty of data — strong models (regularised linear models, gradient boosting) often handle it;
- when you'd select using the **whole dataset** before splitting → leakage (section 14);
- when you'd drop a feature that is required for fairness auditing or business rules.

> 💡 Feature selection is **not guaranteed** to raise accuracy. Its main wins are often simplicity and robustness at *equal* accuracy.

---
<a id="sec-2"></a>
# 2. Feature Selection vs Feature Extraction

| | Feature **selection** | Feature **extraction** |
|---|---|---|
| Output | a subset of the original columns | *new* columns built from all originals |
| Example | keep `alcohol`, `sulphates`, drop the rest | PCA: `PC1 = 0.4·alcohol − 0.3·density + …` |
| Interpretability | high — original names remain | low — components are mixtures |
| Still need to collect all raw columns? | no | yes |

This topic is about **selection**. Extraction (PCA, etc.) is a separate topic; feature *creation* was Topic 11.

---
<a id="sec-3"></a>
# 3. The Three Families of Methods

| Family | How it works | Examples | Speed | Considers feature interactions? | Tied to a model? |
|---|---|---|---|---|---|
| **Filter** | score each feature with a statistic, independent of any model | variance, correlation, ANOVA F, chi², mutual information | ⚡ very fast | ❌ (univariate) | ❌ |
| **Wrapper** | train a model on different subsets and keep the best subset | RFE, RFECV, forward/backward sequential selection | 🐢 slow | ✅ | ✅ |
| **Embedded** | the model selects features *while training* | L1/Lasso, tree-based importance | ⚡/🙂 | ✅ | ✅ |

Plus a **model-agnostic** check: **permutation importance** (shuffle a column, measure the score drop).

A typical practical recipe:
```text
remove junk (constant / duplicate / ID)  →  filter to a manageable number  →  embedded or wrapper method
→ always evaluate the whole thing with cross-validation inside a Pipeline
```

---
<a id="sec-4"></a>
# 4. The Dataset and a Baseline

We use scikit-learn's **Breast Cancer Wisconsin** dataset: 569 tumours, 30 numeric measurements, target = malignant (0) or benign (1).
To make selection meaningful, we **add junk on purpose** — exactly what you meet in real data:

- 10 columns of pure random noise (`noise_0` … `noise_9`);
- a constant column (`clinic_code` = 1 everywhere);
- a quasi-constant column (`rare_flag`, 99% zeros);
- an exact duplicate of a real column (`mean radius copy`).

If a selection method works, it should drop the junk.

In [ ]:
data = load_breast_cancer(as_frame=True)
X_real, y = data.data, data.target

rng = np.random.default_rng(0)
X = X_real.copy()
for i in range(10):
    X[f'noise_{i}'] = rng.normal(size=len(X))
X['clinic_code'] = 1
X['rare_flag'] = (rng.random(len(X)) < 0.01).astype(int)
X['mean radius copy'] = X['mean radius']

JUNK = [f'noise_{i}' for i in range(10)] + ['clinic_code', 'rare_flag', 'mean radius copy']
print('X shape:', X.shape, '| class balance:', y.value_counts(normalize=True).round(3).to_dict())

### Train/test split first
All selection decisions are learned on the **training** set only. The test set is touched once, at the very end.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def cv_score(model, X_, y_=None, scoring='accuracy'):
    y_ = y_train if y_ is None else y_
    s = cross_val_score(model, X_, y_, cv=cv, scoring=scoring)
    return round(float(s.mean()), 4), round(float(s.std()), 4)

baseline = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
print('Baseline, all 43 features   :', cv_score(baseline, X_train))
print('Real 30 features only       :', cv_score(baseline, X_train[X_real.columns]))

The junk costs about **2 percentage points** of CV accuracy (≈0.951 vs ≈0.974) and makes the score less stable
(larger std across folds). Our goal: find the junk **automatically**, and end with a smaller model that is at least as good.

---
<a id="sec-5"></a>
# 5. Filter — Constant and Quasi-Constant Features

### What
A column with (almost) the same value in every row carries (almost) no information.

### How
- `VarianceThreshold(threshold=0)` removes columns with zero variance (constants).
- For quasi-constants, the variance threshold is **scale-dependent** — a variance of 0.01 is tiny for "salary" and huge for
  a feature measured in thousandths. A scale-free check is the **share of the most frequent value**.

### When not
Don't drop a rare-but-important flag blindly (e.g. `is_fraud_history` = 1 for 1% of rows could be very predictive).

In [ ]:
vt = VarianceThreshold(threshold=0)
vt.fit(X_train)
print('Dropped by VarianceThreshold(0):', list(X_train.columns[~vt.get_support()]))

top_share = X_train.apply(lambda s: s.value_counts(normalize=True).iloc[0])
print('\nColumns where one value covers > 95% of rows:')
print(top_share[top_share > 0.95].round(3))

In [ ]:
# Why a raw variance threshold is dangerous: variances of real features differ by orders of magnitude
print(X_train.var().sort_values().head(6).apply(lambda v: f'{v:.2e}'))

Real features such as `fractal dimension` measurements have variances around 10⁻⁵ — **smaller** than the junk `rare_flag`.
`VarianceThreshold(0.01)` would delete useful columns and keep junk. Use threshold 0 for constants and the
*most-frequent-value share* for quasi-constants (or scale with `MinMaxScaler` first).

From here on we remove the constant `clinic_code` (statistical tests cannot score a zero-variance column). We keep
`rare_flag` in for now, to see whether the target-based methods catch it.

In [ ]:
X_train = X_train.drop(columns='clinic_code')
X_test = X_test.drop(columns='clinic_code')
JUNK.remove('clinic_code')
print(X_train.shape)

---
<a id="sec-6"></a>
# 6. Filter — Duplicate and Highly Correlated Features

### Duplicates
Two identical columns add nothing. Find them by transposing and checking duplicated rows, or with a hash per column.

### Highly correlated features
When `|r|` between two features is very high (e.g. > 0.95) they carry nearly the same information. For linear models
this causes **multicollinearity** (unstable coefficients — Topic 12 covers VIF). A simple filter drops one of each pair.

> ⚠️ High correlation between two features is not automatically bad for prediction, and the choice of *which one* to drop
> should prefer the feature that is cheaper, cleaner, or more related to the target.

In [ ]:
dup_cols = X_train.columns[X_train.T.duplicated()]
print('Exact duplicate columns:', list(dup_cols))

def correlated_to_drop(df, threshold=0.95):
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    return [c for c in upper.columns if (upper[c] > threshold).any()]

X_tr_nodup = X_train.drop(columns=dup_cols)
to_drop = correlated_to_drop(X_tr_nodup, 0.95)
print(f'\n{len(to_drop)} features with |r| > 0.95 to an earlier column:')
print(to_drop)

In [ ]:
X_tr_filtered = X_tr_nodup.drop(columns=to_drop)
print('Features left after junk + correlation filter:', X_tr_filtered.shape[1])
print('CV accuracy:', cv_score(baseline, X_tr_filtered))

The correlation filter removes redundant size measurements (radius, perimeter and area of the same tumour are almost
perfectly related) with essentially no loss in accuracy. Note that **noise columns survive** this step — they are not
correlated with anything. Removing them needs a method that looks at the **target** (next section).

---
<a id="sec-7"></a>
# 7. Filter — Univariate Statistical Tests

Score each feature **against the target**, one at a time.

| Score function | Target | Feature | Measures | Notes |
|---|---|---|---|---|
| `f_classif` (ANOVA F) | categorical | numeric | how different the class means are, relative to spread | linear / mean-difference only |
| `chi2` | categorical | **non-negative** (counts, one-hot, min-max scaled) | dependence between feature and class | fails on negative values |
| `mutual_info_classif` | categorical | numeric or discrete | *any* kind of dependence (incl. non-linear) | needs more data; random → set `random_state` |
| `f_regression` | numeric | numeric | linear correlation (as an F statistic) | |
| `mutual_info_regression` | numeric | numeric | any dependence | |

Higher score = more related. `f_classif`, `chi2`, `f_regression` also give **p-values** (small = unlikely to be chance — Topic 14).

In [ ]:
F, p = f_classif(X_train, y_train)
mi = mutual_info_classif(X_train, y_train, random_state=0)
scores = pd.DataFrame({'F': F, 'p_value': p, 'MI': mi}, index=X_train.columns)
scores['F_rank'] = scores['F'].rank(ascending=False).astype(int)
scores['MI_rank'] = scores['MI'].rank(ascending=False).astype(int)
scores.sort_values('F', ascending=False).head(10).round(4)

In [ ]:
print('Where did the junk features rank (out of 42)?')
print(scores.loc[JUNK, ['F_rank', 'MI_rank', 'p_value']].sort_values('F_rank').round(4))

- The top features (worst/mean **concave points**, **perimeter**, **radius**, **area**) are the tumour size/shape measurements.
- The noise columns sink toward the **bottom** of both rankings (ranks 28–42 of 42) — univariate tests catch features
  that have no relation to the target.
- But look at `noise_5`: p ≈ 0.03 < 0.05 **by pure chance**. Test 40 useless features and a couple will look
  "significant" — the *multiple-testing* problem (Topic 14). Use rankings/CV, not a single p < 0.05 cut-off.
- The duplicate `mean radius copy` gets **exactly the same score** as `mean radius`: univariate filters cannot see redundancy.
  That's why section 6 comes first.

In [ ]:
# chi2 needs non-negative input -> min-max scale first
X_pos = MinMaxScaler().fit_transform(X_train)
chi_scores, chi_p = chi2(X_pos, y_train)
print(pd.Series(chi_scores, index=X_train.columns).sort_values(ascending=False).head(5).round(2))

try:
    chi2(X_train[['noise_0']], y_train)       # raw noise has negative values
except ValueError as e:
    print('\nchi2 on negative data ->', e)

---
<a id="sec-8"></a>
# 8. `SelectKBest` and `SelectPercentile`

### Syntax
```python
SelectKBest(score_func=f_classif, k=10)          # keep the 10 best
SelectPercentile(score_func=f_classif, percentile=20)   # keep the best 20%
selector.fit(X_train, y_train)
selector.get_support()          # boolean mask of kept columns
selector.get_feature_names_out()
selector.transform(X_test)      # apply the SAME selection to new data
```

Put the selector **inside a Pipeline**, so it is re-fitted on each training fold (section 14).

In [ ]:
for k in [3, 5, 10, 20, 42]:
    pipe = Pipeline([('scale', StandardScaler()),
                     ('select', SelectKBest(f_classif, k=k)),
                     ('model', LogisticRegression(max_iter=5000))])
    print(f'k={k:>2}:', cv_score(pipe, X_train))

In [ ]:
kbest = SelectKBest(f_classif, k=10).fit(X_train, y_train)
print('10 best by ANOVA F:', list(kbest.get_feature_names_out()))

- Only 3 features already reach ≈0.94. `k=20` scores best here (≈0.977) — **higher than all 42 features** (≈0.951),
  because the noise columns are left out.
- The 10 best features are mostly the same information repeated (radius / perimeter / area, mean and worst) — even the
  duplicate `mean radius copy` is selected. Univariate selection cannot see redundancy; remove correlated features first
  or use a multivariate method.

---
<a id="sec-9"></a>
# 9. Wrapper — Recursive Feature Elimination (RFE / RFECV)

### What
1. Train a model on all features.
2. Rank features by the model's importance (`coef_` for linear models, `feature_importances_` for trees).
3. Remove the weakest (`step` features), retrain, repeat until `n_features_to_select` remain.

**RFECV** does the same but uses cross-validation to **choose the number of features** automatically.

### When
Medium-sized feature sets (tens to a few hundred) where you want a multivariate selection tied to your model.

### When not
Thousands of features (slow — filter first), or with a model that has no `coef_`/`feature_importances_`.

> ⚠️ With linear models, coefficients are only comparable if features are **scaled** → put a scaler before RFE in a Pipeline.

In [ ]:
rfe_pipe = Pipeline([('scale', StandardScaler()),
                     ('rfe', RFE(LogisticRegression(max_iter=5000), n_features_to_select=8, step=2)),
                     ('model', LogisticRegression(max_iter=5000))])
rfe_pipe.fit(X_train, y_train)
rfe_selected = list(X_train.columns[rfe_pipe.named_steps['rfe'].support_])
print('RFE picked:', rfe_selected)
print('junk among them:', [c for c in rfe_selected if c in JUNK])
print('CV accuracy with RFE(8):', cv_score(rfe_pipe, X_train))

In [ ]:
rfecv = RFECV(LogisticRegression(max_iter=5000), step=1, cv=cv, scoring='accuracy', min_features_to_select=1)
rfecv_pipe = Pipeline([('scale', StandardScaler()), ('rfecv', rfecv)])
rfecv_pipe.fit(X_train, y_train)

res = rfecv.cv_results_
print('Optimal number of features:', rfecv.n_features_)
print(pd.DataFrame({'n_features': res['n_features'], 'mean_acc': res['mean_test_score'].round(4),
                    'std': res['std_test_score'].round(4)}).iloc[[0, 1, 2, 4, 7, 11, 19, 29, 41]].to_string(index=False))
plt.figure(figsize=(8, 3.5))
plt.errorbar(res['n_features'], res['mean_test_score'], yerr=res['std_test_score'], marker='o', ms=3, capsize=2)
plt.axvline(rfecv.n_features_, color='red', ls='--', label=f'chosen = {rfecv.n_features_}')
plt.xlabel('number of features'); plt.ylabel('CV accuracy'); plt.legend(); plt.title('RFECV')
plt.show()

The curve rises quickly (≈0.92 with 1 feature, ≈0.96 with 5), peaks around 12 features, and then **declines** as the
junk columns are added back (≈0.95 with all 42) — direct evidence that the noise hurts. Counts from about 5 to 12 are
within roughly one standard deviation of each other: RFECV picks the single highest mean, but a smaller count is often just
as good and simpler. Read the plot, don't just take `n_features_`.

---
<a id="sec-10"></a>
# 10. Wrapper — Sequential Feature Selection

### What
- **Forward:** start with 0 features, add the one that improves the CV score most, repeat.
- **Backward:** start with all, remove the one whose removal hurts least, repeat.

```python
SequentialFeatureSelector(estimator, n_features_to_select=5, direction='forward', cv=5, scoring='accuracy')
```

Unlike RFE it does not need `coef_`, so it works with **any** model — but it's the slowest method
(forward selection of *k* from *p* features trains about *k × p × cv* models).

In [ ]:
sfs_pipe = Pipeline([('scale', StandardScaler()),
                     ('sfs', SequentialFeatureSelector(LogisticRegression(max_iter=5000), n_features_to_select=5,
                                                       direction='forward', cv=3, scoring='accuracy')),
                     ('model', LogisticRegression(max_iter=5000))])
sfs_pipe.fit(X_train, y_train)
sfs_selected = list(X_train.columns[sfs_pipe.named_steps['sfs'].get_support()])
print('Forward selection picked:', sfs_selected)
print('CV accuracy with 5 features:', cv_score(sfs_pipe, X_train))

Forward selection builds a **diverse** set: size (`worst perimeter`, `worst area`) plus texture, smoothness and concavity —
features that complement each other — instead of ten versions of "size" like the univariate top-10. With only 5 features
it matches or beats the full 42-feature model.

---
<a id="sec-11"></a>
# 11. Embedded — L1 Regularisation (Lasso)

### What
An **L1 penalty** adds `λ · Σ|coefficient|` to the loss. It pushes weak coefficients to **exactly zero**, so the model selects
features while it trains. Smaller `C` (logistic) or larger `alpha` (Lasso) = stronger penalty = fewer features.

```python
LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1)   # classification (sklearn ≥ 1.8 style; older: penalty='l1')
Lasso(alpha=0.1) / LassoCV(cv=5)                             # regression
SelectFromModel(model)                                       # keep features with non-zero coefficients
```

### Notes
- **Scale first** — the penalty treats all coefficients equally.
- Among a group of highly correlated features, L1 tends to keep **one** somewhat arbitrarily (see stability, section 16).

In [ ]:
rows = []
for C in [0.01, 0.05, 0.1, 0.5, 1, 10]:
    l1 = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(l1_ratio=1, solver='liblinear', C=C))])
    l1.fit(X_train, y_train)
    coef = pd.Series(l1.named_steps['model'].coef_[0], index=X_train.columns)
    rows.append({'C': C, 'n_nonzero': int((coef != 0).sum()), 'junk_kept': int((coef[JUNK] != 0).sum()),
                 'cv_acc': cv_score(l1, X_train)[0]})
pd.DataFrame(rows)

In [ ]:
l1_sel = Pipeline([('scale', StandardScaler()),
                   ('select', SelectFromModel(LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1))),
                   ('model', LogisticRegression(max_iter=5000))])
l1_sel.fit(X_train, y_train)
l1_selected = list(X_train.columns[l1_sel.named_steps['select'].get_support()])
print(len(l1_selected), 'features kept by L1 (C=0.1):', l1_selected)
print('CV accuracy:', cv_score(l1_sel, X_train))

As `C` decreases, fewer coefficients stay non-zero:
- **large `C`** (weak penalty) keeps many features *including* noise columns and scores lower;
- **`C ≈ 0.1`** keeps about 7 features, **no junk**, and scores *above* the all-feature baseline;
- **tiny `C`** removes too much and accuracy falls.

`C` is a hyper-parameter to tune with CV.

---
<a id="sec-12"></a>
# 12. Embedded — Tree-Based Importance

Random forests and gradient boosting report `feature_importances_` = how much each feature reduced impurity across all splits.

| ✅ Pros | ❌ Cons |
|---|---|
| no scaling needed, captures non-linear effects and interactions | **biased toward high-cardinality / continuous features** — noise columns get non-zero importance |
| free by-product of training | importance is split between correlated features |
| | computed on **training** data → reflects what the model *used*, not what *generalises* |

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(8).round(4))
print('\nImportance of the noise features:')
print(imp[[c for c in JUNK if c.startswith('noise')]].round(4).sort_values(ascending=False).head(5))

In [ ]:
rf_sel = SelectFromModel(RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1), threshold='median')
rf_pipe = Pipeline([('select', rf_sel), ('model', LogisticRegression(max_iter=5000))])
rf_pipe.fit(X_train, y_train)
rf_selected = list(X_train.columns[rf_pipe.named_steps['select'].get_support()])
print(len(rf_selected), 'features above median importance; junk among them:', [c for c in rf_selected if c in JUNK])

Pure-noise columns receive small but **non-zero** importance — the forest can always find a few "useful-looking" splits in
random numbers on the training set. Impurity importance ranks features; it cannot tell you a feature is useless.
`threshold='median'` keeps the top half.

---
<a id="sec-13"></a>
# 13. Permutation Importance (model-agnostic)

### What
1. Train any model. 2. Measure its score on **held-out** data. 3. Shuffle one column (breaking its link with the target),
re-score. 4. Importance = score drop. Repeat the shuffle several times (`n_repeats`) to get a mean ± std.

### Why it's often better than impurity importance
- computed on held-out data → measures what **generalises**; useless features get ≈ 0 (can be slightly negative);
- works for any model.

### Caveat
With **correlated** features, shuffling one barely hurts because its twin still carries the information → both look
unimportant. Group or de-correlate features first (section 6).

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.3, stratify=y_train, random_state=1)
rf_perm = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(X_fit, y_fit)
# scoring='neg_log_loss': with only ~130 validation rows, accuracy moves in coarse steps (1 row ≈ 0.8%);
# log-loss reacts to changes in predicted probabilities, so drops are much easier to see.
perm = permutation_importance(rf_perm, X_val, y_val, n_repeats=20, random_state=0, n_jobs=-1, scoring='neg_log_loss')
perm_df = pd.DataFrame({'mean': perm.importances_mean, 'std': perm.importances_std},
                       index=X_train.columns).sort_values('mean', ascending=False)
print(perm_df.head(8).round(4))
print('\nNoise features:')
print(perm_df.loc[[c for c in JUNK if c.startswith('noise')]].round(4).sort_values('mean', ascending=False).head(5))

In [ ]:
top = perm_df.head(12).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(top.index, top['mean'], xerr=top['std'])
plt.title('Permutation importance (log-loss increase on validation data)')
plt.tight_layout(); plt.show()

The ranking is led by concave-points and size features. Noise features sit at ≈ **zero** (or slightly negative = shuffling
them doesn't hurt at all). Also notice that importance is **shared** among the correlated size family (area, radius, perimeter):
each one looks only moderately important because its twins still carry the information.

---
<a id="sec-14"></a>
# 14. ⚠️ Selection Leakage — the Most Important Section

### The mistake
Selecting features on the **full** dataset (using the target), then cross-validating a model on those features.
The selection has already "seen" the validation folds' targets, so the CV score is **optimistic** — sometimes wildly.

### Demonstration on pure noise
2,000 random features, a random target, 100 rows. The honest accuracy is ~50% (coin flip).

In [ ]:
rng = np.random.default_rng(42)
X_noise = rng.normal(size=(100, 2000))
y_noise = rng.integers(0, 2, 100)
cv_noise = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: select on all rows first, then cross-validate
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=5000), X_sel, y_noise, cv=cv_noise).mean()

# RIGHT: selection inside the pipeline -> refitted on each training fold only
right_pipe = Pipeline([('select', SelectKBest(f_classif, k=20)), ('model', LogisticRegression(max_iter=5000))])
right = cross_val_score(right_pipe, X_noise, y_noise, cv=cv_noise).mean()

print(f'WRONG (select then CV): {wrong:.3f}')
print(f'RIGHT (select inside CV): {right:.3f}')

The wrong workflow reports impressive accuracy on data that contains **no signal at all**. With 2,000 random features,
some will correlate with the target by chance; selecting them using all rows bakes that chance into every fold.

**Rule:** any step that looks at the target — univariate tests, RFE, L1, importance thresholds — must be *inside* the
Pipeline that is cross-validated, and the final test set must never influence selection.

---
<a id="sec-15"></a>
# 15. Tuning the Number of Features with a Pipeline

Treat `k` (or `C`, or a threshold) as a **hyper-parameter** and let `GridSearchCV` choose it — selection stays inside each fold.

In [ ]:
pipe = Pipeline([('scale', StandardScaler()),
                 ('select', SelectKBest(f_classif)),
                 ('model', LogisticRegression(max_iter=5000))])
grid = GridSearchCV(pipe, {'select__k': [2, 4, 6, 8, 10, 15, 20, 30, 42]}, cv=cv, scoring='accuracy')
grid.fit(X_train, y_train)

res = pd.DataFrame(grid.cv_results_)[['param_select__k', 'mean_test_score', 'std_test_score']].round(4)
print(res.to_string(index=False))
print('\nbest k:', grid.best_params_['select__k'], '| best CV accuracy:', round(grid.best_score_, 4))

In [ ]:
# The "one-standard-error rule": the smallest k whose score is within 1 std of the best
best = res.loc[res['mean_test_score'].idxmax()]
ok = res[res['mean_test_score'] >= best['mean_test_score'] - best['std_test_score']]
print('smallest k within one std of the best:', int(ok['param_select__k'].min()))

Here the band "best score − 1 std" only contains `k = 20` and `k = 30`, so the rule agrees with the best `k`.
On many datasets the curve is flat and the rule picks a **much smaller** `k` for almost the same score — prefer the simpler model.
Note the dip at `k = 15`: CV curves are noisy, so don't over-interpret single points.

---
<a id="sec-16"></a>
# 16. Stability — Would You Pick the Same Features Again?

A selection that changes completely when a few rows change is not trustworthy. Check it by re-running the selection on
**bootstrap samples** (random rows drawn with replacement) and counting how often each feature is chosen.

In [ ]:
l1_model = Pipeline([('scale', StandardScaler()),
                     ('model', LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1))])
counts = pd.Series(0, index=X_train.columns)
rng = np.random.default_rng(0)
B = 30
for b in range(B):
    idx = rng.integers(0, len(X_train), len(X_train))
    l1_model.fit(X_train.iloc[idx], y_train.iloc[idx])
    counts += (l1_model.named_steps['model'].coef_[0] != 0)

freq = (counts / B).sort_values(ascending=False)
print('Selected in >= 80% of bootstraps:', list(freq[freq >= 0.8].index))
print('\nSelected in 20-80% (unstable):', list(freq[(freq > 0.2) & (freq < 0.8)].index))
print('\nMax selection frequency of any noise feature:', freq[[c for c in JUNK if c.startswith('noise')]].max())

- A **core** of features is selected almost every time — these are reliable.
- A group sits in the middle: typically members of a correlated family where L1 picks "one of them", differently each time.
- Noise features are selected rarely, if ever.

Report stable features with confidence; treat the unstable ones as interchangeable.

---
<a id="sec-17"></a>
# 17. Categorical Features

| Situation | Method |
|---|---|
| categorical feature, categorical target | `chi2` on one-hot columns · `mutual_info_classif(discrete_features=True)` |
| categorical feature, numeric target | ANOVA (`f_regression` on one-hot) · `mutual_info_regression(discrete_features=True)` |
| many one-hot columns from one feature | score each dummy, then **aggregate by original feature** (sum/max) to decide keep/drop for the whole feature |

Selecting individual dummy columns (keeping `City_Lahore` but dropping `City_Karachi`) is allowed but makes the model harder to explain.

In [ ]:
rng = np.random.default_rng(3)
n = 800
cat_df = pd.DataFrame({
    'Contract': rng.choice(['Monthly', 'Yearly', 'TwoYear'], n, p=[0.5, 0.3, 0.2]),
    'Payment': rng.choice(['Card', 'Cash', 'Transfer', 'Online'], n),
    'Gender': rng.choice(['M', 'F'], n),
    'Region': rng.choice(['North', 'South', 'East', 'West', 'Central'], n),
})
p_churn = cat_df['Contract'].map({'Monthly': 0.40, 'Yearly': 0.12, 'TwoYear': 0.06}) + (cat_df['Payment'] == 'Cash') * 0.10
y_cat = (rng.random(n) < p_churn).astype(int)

ohe = OneHotEncoder(sparse_output=False)
X_cat = ohe.fit_transform(cat_df)
names = ohe.get_feature_names_out()
chi_s, chi_p = chi2(X_cat, y_cat)
mi_s = mutual_info_classif(X_cat, y_cat, discrete_features=True, random_state=0)
dummy_scores = pd.DataFrame({'chi2': chi_s, 'p': chi_p, 'MI': mi_s}, index=names)
dummy_scores['feature'] = [nm.split('_')[0] for nm in names]
print(dummy_scores.round(4).sort_values('chi2', ascending=False).head(6))
print('\nAggregated by original feature:')
print(dummy_scores.groupby('feature')[['chi2', 'MI']].sum().sort_values('chi2', ascending=False).round(4))

`Contract` (which drives churn in the simulation) dominates, `Payment` comes second (the *Cash* effect), and `Gender` / `Region`
score near zero — as designed.

---
<a id="sec-18"></a>
# 18. Comparing Methods and Choosing One

Let's compare the feature sets found above and evaluate the chosen approach **once** on the untouched test set.

In [ ]:
kbest_sel = list(SelectKBest(f_classif, k=8).fit(X_train, y_train).get_feature_names_out())
sets = {'ANOVA top-8': kbest_sel, 'RFE(8)': rfe_selected, 'Forward SFS(5)': sfs_selected,
        'L1 (C=0.1)': l1_selected, 'RF > median': rf_selected}
sets['All features'] = list(X_train.columns)
comp = pd.DataFrame({name: {'n_features': len(f),
                            'junk_kept': len([c for c in f if c in JUNK]),
                            'cv_acc': cv_score(baseline, X_train[f])[0]} for name, f in sets.items()}).T
comp.astype({'n_features': int, 'junk_kept': int})

> ⚠️ The `cv_acc` column above re-uses features that were selected on the *whole training set*, so it is slightly
> optimistic (section 14). It's fine for a rough comparison; the honest numbers are the pipeline scores printed in each section.

In [ ]:
final = Pipeline([('scale', StandardScaler()),
                  ('select', SelectFromModel(LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1))),
                  ('model', LogisticRegression(max_iter=5000))])
final.fit(X_train, y_train)
kept = X_train.columns[final.named_steps['select'].get_support()]
print(f'Final model uses {len(kept)} of {X_train.shape[1]} features')
print('Test accuracy (touched once):', round(final.score(X_test, y_test), 4))
print('Baseline test accuracy      :', round(baseline.fit(X_train, y_train).score(X_test, y_test), 4))
print('Test set size:', len(X_test))

On the 143 untouched test tumours, the 7-feature model and the 42-feature baseline differ by **about one patient**
(≈0.7 percentage points) — well within noise for a test set this small. Same accuracy with **6× fewer inputs and none of
the junk**: the typical win of feature selection. (Cross-validation, which averages over more data, favoured the selected model.)

### Decision guide

| Situation | Good first choice |
|---|---|
| obvious junk (constant, duplicate, ID) | drop manually / `VarianceThreshold(0)` |
| thousands of features | univariate filter (`SelectKBest` with F or MI) to a few hundred, then embedded |
| linear model, want sparsity | **L1** + `SelectFromModel`, tune `C` |
| tree model | permutation importance on validation data; `SelectFromModel` with a threshold |
| tens of features, want the best subset for *this* model | RFECV or sequential selection |
| need to explain which inputs matter | L1 or permutation importance + stability check |
| correlated groups | de-correlate first (section 6 / Topic 12) or treat the group as a unit |

---
<a id="sec-19"></a>
# 19. Common Mistakes

| Mistake | Consequence | Fix |
|---|---|---|
| Selecting on the full dataset, then cross-validating | optimistic, sometimes fake, scores (section 14) | selector inside a `Pipeline` |
| Using the test set to pick features or `k` | test score no longer honest | tune on CV within the training set only |
| `VarianceThreshold(0.01)` on unscaled data | drops real small-scale features, keeps junk | threshold 0, or most-frequent-share, or scale first |
| `chi2` on negative values | `ValueError` | min-max scale or use counts/one-hot |
| Forgetting to scale before L1 / RFE with linear models | coefficient sizes reflect units, not importance | `StandardScaler` first |
| Trusting impurity importance of trees | noise columns look "important" | permutation importance on held-out data |
| Removing a feature because its univariate score is low | misses features useful only in combination | also try a multivariate method |
| Reading one run of L1/RFE as the truth | selection can be unstable | bootstrap stability check |
| Dropping all but one of a correlated group without thinking | may drop the cleaner / cheaper feature | choose deliberately |
| Expecting selection to always raise accuracy | disappointment | the main win is often a simpler model at equal accuracy |

---
# Practice Problems (18, increasing difficulty)

The practice uses a **regression** problem, so you meet the regression versions of every tool:
scikit-learn's **diabetes** dataset (442 patients, 10 standardised measurements, target = disease progression after one year).
We add junk columns just like in the lesson.

Run the next cell to create `Xd`, `yd` and the train/test split. Each problem has an empty cell; solutions are at the end.

In [ ]:
diab = load_diabetes(as_frame=True)
Xd, yd = diab.data.copy(), diab.target

rng_p = np.random.default_rng(11)
for i in range(5):
    Xd[f'rand_{i}'] = rng_p.normal(0, 0.05, len(Xd))          # noise on the same scale as the real features
Xd['site'] = 1                                                  # constant
Xd['vip_flag'] = (rng_p.random(len(Xd)) < 0.02).astype(int)    # quasi-constant
Xd['bmi_dup'] = Xd['bmi']                                       # duplicate

Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.25, random_state=0)
kf = KFold(n_splits=5, shuffle=True, random_state=0)
print(Xd.shape, '| real features:', list(diab.data.columns))

### ✍️ Practice Problem 1

**(Easy) Baseline:** Cross-validate a `LinearRegression` on all columns of `Xd_train` with `kf` and `scoring='r2'`. Report the mean and std R².

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 1 — write your solution here

### ✍️ Practice Problem 2

**(Easy) Constant columns:** Use `VarianceThreshold(0)` to find constant columns in `Xd_train`.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 2 — write your solution here

### ✍️ Practice Problem 3

**(Easy) Quasi-constant — the scale trap:** Print the variance of every column. Explain why `VarianceThreshold(0.01)` would be a terrible choice here. Then find quasi-constant columns with the *most-frequent-value share* (> 95%).

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 3 — write your solution here

### ✍️ Practice Problem 4

**(Easy) Duplicates:** Find exactly duplicated columns.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 4 — write your solution here

### ✍️ Practice Problem 5

**(Medium) Correlated pairs:** List all pairs of real features with `|r| > 0.8` in `Xd_train`. Which would you drop?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 5 — write your solution here

### ✍️ Practice Problem 6

**(Medium) `f_regression`:** Score all columns (except the constant) with `f_regression`. Show F, p-value and rank. Where do the `rand_*` columns rank?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 6 — write your solution here

### ✍️ Practice Problem 7

**(Medium) Mutual information:** Score the same columns with `mutual_info_regression(random_state=0)`. Compare the top 5 with the F-test top 5.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 7 — write your solution here

### ✍️ Practice Problem 8

**(Medium) Tune k:** Build `Pipeline([VarianceThreshold(0), SelectKBest(f_regression), LinearRegression()])` and use `GridSearchCV` over `k = 1…15` with `kf`, `scoring='r2'`. What is the best `k`?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 8 — write your solution here

### ✍️ Practice Problem 9

**(Medium) RFE:** Use `RFE(LinearRegression(), n_features_to_select=5)` inside a pipeline with a scaler. Which 5 features are chosen? Any junk?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 9 — write your solution here

### ✍️ Practice Problem 10

**(Medium) RFECV:** Run `RFECV(LinearRegression(), cv=kf, scoring='r2')` (after dropping the constant). How many features does it choose? Plot the CV curve.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 10 — write your solution here

### ✍️ Practice Problem 11

**(Medium) Forward selection:** Use `SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='forward', cv=kf, scoring='r2')`. Which features are added?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 11 — write your solution here

### ✍️ Practice Problem 12

**(Hard) Lasso:** Fit `LassoCV(cv=kf, random_state=0)` on scaled training data (inside a pipeline). Which coefficients are exactly zero? What `alpha` was chosen?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 12 — write your solution here

### ✍️ Practice Problem 13

**(Hard) Tree vs permutation importance:** Train a `RandomForestRegressor(n_estimators=300, random_state=0)` on `Xd_train`. Compare impurity importance with permutation importance on `Xd_test` (`n_repeats=20`). Where do the `rand_*` columns land in each?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 13 — write your solution here

### ✍️ Practice Problem 14

**(Hard) Compare methods:** Make a table with one row per real + junk feature and one column per method (F top-5, MI top-5, RFE-5, SFS-4, Lasso non-zero) marking selected with 1. Which features does every method agree on?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 14 — write your solution here

### ✍️ Practice Problem 15

**(Hard) Leakage demo (regression):** Create `Xn = rng.normal(size=(80, 1000))` and `yn = rng.normal(size=80)` (pure noise). Compare CV R² when `SelectKBest(f_regression, k=10)` is fitted on all rows *before* CV vs. inside a pipeline.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 15 — write your solution here

### ✍️ Practice Problem 16

**(Hard) Stability:** On 30 bootstrap samples of the training set, fit a `Lasso(alpha=5.0)` pipeline (scaler + Lasso) and count how often each feature has a non-zero coefficient. Which features form a stable core?

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 16 — write your solution here

### ✍️ Practice Problem 17

**(Hard) Categorical selection:** Using `pd.cut(yd, 3, labels=['low', 'mid', 'high'])` as a class target and the diabetes `sex` column plus a random 4-level category, score the one-hot columns with `chi2` and `mutual_info_classif(discrete_features=True)`. Aggregate by original feature.

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 17 — write your solution here

### ✍️ Practice Problem 18

**(Hard) Final model:** Build the best pipeline you can with feature selection inside it (e.g. scaler → `SelectFromModel(LassoCV)` → `LinearRegression`, or a tuned `SelectKBest`). Report CV R² vs baseline, the final features, and the **test** R² (touched once).

*(Solution in the **Practice Solutions** section at the end.)*

In [ ]:
# ✍️ Practice Problem 18 — write your solution here

---
# Practice Solutions

Run them top to bottom (some re-use results from earlier solutions).

### Solution 1 — Baseline

In [ ]:
def cv_r2(model, X_, y_=None):
    s = cross_val_score(model, X_, yd_train if y_ is None else y_, cv=kf, scoring='r2')
    return round(float(s.mean()), 4), round(float(s.std()), 4)

print('Baseline R² (all 18 columns):', cv_r2(LinearRegression(), Xd_train))
print('Real 10 features only       :', cv_r2(LinearRegression(), Xd_train[diab.data.columns]))

### Solution 2 — Constant columns

In [ ]:
vt = VarianceThreshold(0).fit(Xd_train)
print(list(Xd_train.columns[~vt.get_support()]))

### Solution 3 — Quasi-constant and the scale trap

In [ ]:
print(Xd_train.var().sort_values().round(5))
share = Xd_train.apply(lambda s: s.value_counts(normalize=True).iloc[0])
print('\nmost-frequent share > 95%:', share[share > 0.95].round(3).to_dict())

The real diabetes features are pre-scaled and have variances of only ≈ 0.002, while the junk `vip_flag` has a variance
around 0.02. `VarianceThreshold(0.01)` would delete **every real feature** and keep the junk flag. The most-frequent-value
share is scale-free and flags `site` (100%) and `vip_flag` (~98%).

### Solution 4 — Duplicates

In [ ]:
print(list(Xd_train.columns[Xd_train.T.duplicated()]))

### Solution 5 — Correlated pairs

In [ ]:
real = list(diab.data.columns)
corr = Xd_train[real].corr()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print(pairs[pairs.abs() > 0.8].round(3))

`s1` (total cholesterol) and `s2` (LDL) are strongly correlated (≈ 0.9). Keep one — e.g. the one with the stronger relation to
the target or the one easier to measure — or keep both but use a regularised model (Ridge/Lasso). See VIF in Topic 12.

### Solution 6 — `f_regression`

In [ ]:
Xd_nc = Xd_train.drop(columns='site')
Fr, pr = f_regression(Xd_nc, yd_train)
freg = pd.DataFrame({'F': Fr, 'p': pr}, index=Xd_nc.columns)
freg['rank'] = freg['F'].rank(ascending=False).astype(int)
print(freg.sort_values('F', ascending=False).round(4))

### Solution 7 — Mutual information

In [ ]:
mi_r = pd.Series(mutual_info_regression(Xd_nc, yd_train, random_state=0), index=Xd_nc.columns).sort_values(ascending=False)
print(mi_r.round(4).head(8))
print('\nF top-5 :', list(freg.sort_values('F', ascending=False).index[:5]))
print('MI top-5:', list(mi_r.index[:5]))

### Solution 8 — Tune k

In [ ]:
pipe_k = Pipeline([('const', VarianceThreshold(0)), ('select', SelectKBest(f_regression)), ('model', LinearRegression())])
gs_k = GridSearchCV(pipe_k, {'select__k': list(range(1, 16))}, cv=kf, scoring='r2').fit(Xd_train, yd_train)
res_k = pd.DataFrame(gs_k.cv_results_)[['param_select__k', 'mean_test_score', 'std_test_score']].round(4)
print(res_k.to_string(index=False))
print('best k:', gs_k.best_params_['select__k'], '| CV R²:', round(gs_k.best_score_, 4))

### Solution 9 — RFE

In [ ]:
rfe_d = Pipeline([('const', VarianceThreshold(0)), ('scale', StandardScaler()),
                  ('rfe', RFE(LinearRegression(), n_features_to_select=5)), ('model', LinearRegression())])
rfe_d.fit(Xd_train, yd_train)
cols_nc = Xd_train.columns[rfe_d.named_steps['const'].get_support()]
rfe5 = list(cols_nc[rfe_d.named_steps['rfe'].support_])
print('RFE-5:', rfe5)
print('CV R²:', cv_r2(rfe_d, Xd_train))

RFE with a linear model ranks by |coefficient|. It keeps **both** `s1` and `s2` (correlated ≈ 0.89): together they get large
opposite coefficients — a side-effect of multicollinearity (Topic 12). It also keeps both `bmi` and its duplicate, which
wastes one of the 5 slots. Lesson: remove duplicates/near-duplicates **before** a wrapper method.

### Solution 10 — RFECV

In [ ]:
rfecv_d = RFECV(LinearRegression(), cv=kf, scoring='r2').fit(Xd_nc, yd_train)
print('chosen number of features:', rfecv_d.n_features_)
print('features:', list(Xd_nc.columns[rfecv_d.support_]))
r = rfecv_d.cv_results_
plt.figure(figsize=(7, 3))
plt.errorbar(r['n_features'], r['mean_test_score'], yerr=r['std_test_score'], marker='o', ms=3, capsize=2)
plt.xlabel('number of features'); plt.ylabel('CV R²'); plt.title('RFECV — diabetes'); plt.show()

### Solution 11 — Forward selection

In [ ]:
sfs_d = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4, direction='forward', cv=kf, scoring='r2')
sfs_d.fit(Xd_nc, yd_train)
sfs4 = list(Xd_nc.columns[sfs_d.get_support()])
print('SFS-4:', sfs4)
print('CV R² with these 4:', cv_r2(LinearRegression(), Xd_train[sfs4]))

### Solution 12 — Lasso

In [ ]:
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', LassoCV(cv=kf, random_state=0))]).fit(Xd_nc, yd_train)
coefs = pd.Series(lasso_pipe.named_steps['lasso'].coef_, index=Xd_nc.columns)
print('alpha chosen:', round(lasso_pipe.named_steps['lasso'].alpha_, 4))
print('zero coefficients:', list(coefs[coefs == 0].index))
print(coefs[coefs != 0].round(2).sort_values(key=abs, ascending=False))
lasso_nonzero = list(coefs[coefs != 0].index)

- Lasso puts almost all the weight of the duplicated pair on one column (`bmi` large, `bmi_dup` tiny).
- `LassoCV` chooses `alpha` for the best **prediction**, not for the sparsest model, so one noise column (`rand_4`) survives
  with a small coefficient. Stability (Solution 16) is how you find out which survivors to trust.

### Solution 13 — Tree vs permutation importance

In [ ]:
rf_d = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1).fit(Xd_train, yd_train)
imp_d = pd.DataFrame({'impurity': rf_d.feature_importances_}, index=Xd_train.columns)
pi = permutation_importance(rf_d, Xd_test, yd_test, n_repeats=20, random_state=0, n_jobs=-1)
imp_d['permutation'] = pi.importances_mean
imp_d['impurity_rank'] = imp_d['impurity'].rank(ascending=False).astype(int)
imp_d['perm_rank'] = imp_d['permutation'].rank(ascending=False).astype(int)
print(imp_d.sort_values('impurity', ascending=False).round(4))

- **Impurity** importance gives every `rand_*` column a clearly non-zero value (≈0.03–0.04) — `rand_1` even ranks above
  real features such as `s6`, `s2`, `age` and `s1`. Continuous random numbers offer many split points to overfit.
- **Permutation** importance puts most noise columns near zero or below — but not all: on only 111 test rows, `rand_4`
  gets a positive score by chance. Permutation importance is honest but **noisy** on small data (look at the std, use more repeats or CV).
- `bmi` gets ≈ 0 permutation importance although it is the strongest feature! Its duplicate `bmi_dup` still carries the
  information when `bmi` is shuffled — the correlated-feature caveat from section 13.

### Solution 14 — Compare methods

In [ ]:
f5 = list(freg.sort_values('F', ascending=False).index[:5])
mi5 = list(mi_r.index[:5])
methods = {'F top-5': f5, 'MI top-5': mi5, 'RFE-5': rfe5, 'SFS-4': sfs4, 'Lasso≠0': lasso_nonzero}
table = pd.DataFrame({m: Xd_nc.columns.isin(f).astype(int) for m, f in methods.items()}, index=Xd_nc.columns)
table['votes'] = table.sum(axis=1)
print(table.sort_values('votes', ascending=False))
print('\nChosen by every method:', list(table.index[table['votes'] == len(methods)]))

### Solution 15 — Leakage demo (regression)

In [ ]:
rng_l = np.random.default_rng(5)
Xn, yn = rng_l.normal(size=(80, 1000)), rng_l.normal(size=80)
kf_n = KFold(5, shuffle=True, random_state=0)
X_leak = SelectKBest(f_regression, k=10).fit_transform(Xn, yn)
wrong = cross_val_score(LinearRegression(), X_leak, yn, cv=kf_n, scoring='r2').mean()
right = cross_val_score(Pipeline([('sel', SelectKBest(f_regression, k=10)), ('lr', LinearRegression())]),
                        Xn, yn, cv=kf_n, scoring='r2').mean()
print(f'WRONG (select then CV): R² = {wrong:.3f}')
print(f'RIGHT (inside CV)     : R² = {right:.3f}')

On pure noise the honest R² is ≤ 0 (worse than predicting the mean). Selecting on all rows first produces a clearly
positive R² — a model that "explains" random numbers. That's selection leakage.

### Solution 16 — Stability

In [ ]:
las = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=5.0))])
cnt = pd.Series(0, index=Xd_nc.columns)
rng_b = np.random.default_rng(0)
for _ in range(30):
    idx = rng_b.integers(0, len(Xd_nc), len(Xd_nc))
    las.fit(Xd_nc.iloc[idx], yd_train.iloc[idx])
    cnt += (las.named_steps['lasso'].coef_ != 0)
print((cnt / 30).sort_values(ascending=False).round(2))

A **stable core** (`bmi`/`bmi_dup`, `s5`, `bp`, `s3`) is selected in every bootstrap. `sex` is borderline (about half),
and the noise columns appear only occasionally (at most about a third of the time) — so a single Lasso run that kept a
noise column (as in Solution 12) should not be trusted on that point.

### Solution 17 — Categorical selection

In [ ]:
y_cls = pd.cut(yd, 3, labels=['low', 'mid', 'high'])
cat_p = pd.DataFrame({'sex': np.where(Xd['sex'] > 0, 'A', 'B'),
                      'random_cat': np.random.default_rng(2).choice(list('WXYZ'), len(Xd))})
ohe_p = OneHotEncoder(sparse_output=False).fit(cat_p)
Xc = ohe_p.transform(cat_p)
nm = ohe_p.get_feature_names_out()
sc = pd.DataFrame({'chi2': chi2(Xc, y_cls)[0], 'p': chi2(Xc, y_cls)[1],
                   'MI': mutual_info_classif(Xc, y_cls, discrete_features=True, random_state=0)}, index=nm)
sc['feature'] = [n.rsplit('_', 1)[0] for n in nm]
print(sc.round(4))
print(sc.groupby('feature')[['chi2', 'MI']].sum().round(4))

Both scores are small for both features — in this dataset `sex` alone has little relation to the progression class, and the
random category has none. Small chi² with large p-values → no evidence of dependence. (Read the p-values rather than the raw chi² size.)

### Solution 18 — Final model

In [ ]:
final_d = Pipeline([('const', VarianceThreshold(0)), ('scale', StandardScaler()),
                    ('select', SelectFromModel(LassoCV(cv=kf, random_state=0))),
                    ('model', LinearRegression())])
print('baseline CV R²      :', cv_r2(LinearRegression(), Xd_train))
print('selection pipeline  :', cv_r2(final_d, Xd_train))
final_d.fit(Xd_train, yd_train)
cols_after_const = Xd_train.columns[final_d.named_steps['const'].get_support()]
final_feats = list(cols_after_const[final_d.named_steps['select'].get_support()])
print(f'{len(final_feats)} final features:', final_feats)
print('test R² (touched once):', round(final_d.score(Xd_test, yd_test), 4),
      '| baseline test R²:', round(LinearRegression().fit(Xd_train, yd_train).score(Xd_test, yd_test), 4))

**Honest reading:** the selection pipeline and the baseline have practically the same CV R² (difference far smaller than the
fold-to-fold std of ≈ 0.08), and on the 111-row test set the selected model is slightly *lower*. Feature selection here gives a
**simpler model with about the same performance**, not a better score — which is a perfectly normal outcome. The test R² is
lower than the CV R² for both models: this particular test split is harder, a reminder that one split is a noisy estimate.

---
<a id="sec-20"></a>
# 🚀 Project — Feature Selection for Red-Wine Quality

**Data:** `data/winequality-red.csv` (the dataset explored in Topic 7's project).
**Task:** predict whether a wine is **good** (`quality ≥ 7`, ≈14% of wines) from 11 lab measurements.
**Goal:** find the smallest, most reliable feature set that predicts as well as all 11, using a leakage-free workflow.

Because good wines are the minority, we score with **ROC-AUC** (Topic 17) instead of accuracy.

### Plan
1. Load, drop duplicates (as decided in Topic 7), split train/test (stratified).
2. Baseline with all features.
3. Filter view: ANOVA F and mutual information.
4. Embedded: L1 logistic regression — tune `C`.
5. Wrapper: RFECV.
6. Model-agnostic: permutation importance of a random forest.
7. Stability of the L1 selection.
8. Final pipeline, honest CV, one test-set evaluation, conclusions.

In [ ]:
wine = pd.read_csv('data/winequality-red.csv').drop_duplicates().reset_index(drop=True)
Xw = wine.drop(columns='quality')
yw = (wine['quality'] >= 7).astype(int)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.25, stratify=yw, random_state=42)
cv_w = StratifiedKFold(5, shuffle=True, random_state=0)
print(Xw.shape, '| share of good wines:', round(yw.mean(), 3))

def auc(model, X_):
    s = cross_val_score(model, X_, yw_train, cv=cv_w, scoring='roc_auc')
    return round(float(s.mean()), 4), round(float(s.std()), 4)

lr_w = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
print('Baseline (11 features) CV ROC-AUC:', auc(lr_w, Xw_train))

### Step 3 · Filter scores

In [ ]:
Fw, pw = f_classif(Xw_train, yw_train)
filt = pd.DataFrame({'F': Fw, 'p': pw, 'MI': mutual_info_classif(Xw_train, yw_train, random_state=0)},
                    index=Xw_train.columns)
filt['F_rank'] = filt['F'].rank(ascending=False).astype(int)
filt['MI_rank'] = filt['MI'].rank(ascending=False).astype(int)
filt.sort_values('F', ascending=False).round(4)

### Step 4 · Embedded — L1 logistic regression, tuned `C`

In [ ]:
l1_w = Pipeline([('scale', StandardScaler()),
                 ('model', LogisticRegression(l1_ratio=1, solver='liblinear'))])
gs_w = GridSearchCV(l1_w, {'model__C': [0.005, 0.01, 0.02, 0.05, 0.1, 0.3, 1, 10]}, cv=cv_w, scoring='roc_auc')
gs_w.fit(Xw_train, yw_train)
rows = []
for C, sc, sd in zip(gs_w.cv_results_['param_model__C'], gs_w.cv_results_['mean_test_score'], gs_w.cv_results_['std_test_score']):
    m = LogisticRegression(l1_ratio=1, solver='liblinear', C=C).fit(StandardScaler().fit_transform(Xw_train), yw_train)
    rows.append({'C': C, 'n_nonzero': int((m.coef_ != 0).sum()), 'cv_auc': round(sc, 4), 'std': round(sd, 4)})
pd.DataFrame(rows)

The best AUC comes from a fairly large `C` that keeps 10–11 features — the L1 penalty barely acts. But the AUC curve is almost
flat from `C = 0.1` upward (differences of ~0.001, far below the fold std of ~0.04). We therefore choose the **simplest model whose CV AUC is within 0.005 of the best** (a practical
tolerance; the stricter one-standard-error rule would keep only 3 features at a visibly lower AUC).

In [ ]:
res_w = pd.DataFrame(rows)
chosen_C = res_w.loc[res_w['cv_auc'] >= res_w['cv_auc'].max() - 0.005, 'C'].min()
one_se_C = res_w.loc[res_w['cv_auc'] >= res_w['cv_auc'].max() - res_w.loc[res_w['cv_auc'].idxmax(), 'std'], 'C'].min()
print('best C:', gs_w.best_params_['model__C'], '| chosen C (within 0.005):', chosen_C, '| one-SE C:', one_se_C)

l1_chosen = Pipeline([('scale', StandardScaler()),
                      ('model', LogisticRegression(l1_ratio=1, solver='liblinear', C=chosen_C))]).fit(Xw_train, yw_train)
coef_w = pd.Series(l1_chosen.named_steps['model'].coef_[0], index=Xw_train.columns)
print(coef_w.round(3).sort_values(key=abs, ascending=False))

### Step 5 · Wrapper — RFECV

In [ ]:
rfecv_w = Pipeline([('scale', StandardScaler()),
                    ('rfecv', RFECV(LogisticRegression(max_iter=5000), cv=cv_w, scoring='roc_auc'))])
rfecv_w.fit(Xw_train, yw_train)
rf_step = rfecv_w.named_steps['rfecv']
print('RFECV chooses', rf_step.n_features_, 'features:', list(Xw_train.columns[rf_step.support_]))
r = rf_step.cv_results_
print(pd.DataFrame({'n': r['n_features'], 'auc': r['mean_test_score'].round(4), 'std': r['std_test_score'].round(4)}).to_string(index=False))

### Step 6 · Permutation importance of a random forest

In [ ]:
Xf_w, Xv_w, yf_w, yv_w = train_test_split(Xw_train, yw_train, test_size=0.3, stratify=yw_train, random_state=1)
rf_w = RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1).fit(Xf_w, yf_w)
pi_w = permutation_importance(rf_w, Xv_w, yv_w, n_repeats=30, random_state=0, scoring='roc_auc', n_jobs=-1)
perm_w = pd.DataFrame({'mean': pi_w.importances_mean, 'std': pi_w.importances_std},
                      index=Xw_train.columns).sort_values('mean', ascending=False)
print(perm_w.round(4))
perm_w['mean'].iloc[::-1].plot(kind='barh', xerr=perm_w['std'].iloc[::-1], figsize=(7, 4),
                                 title='Permutation importance (ROC-AUC drop)')
plt.show()

### Step 7 · Stability of the L1 selection

In [ ]:
best_C = chosen_C
cnt_w = pd.Series(0, index=Xw_train.columns)
rng_w = np.random.default_rng(0)
for _ in range(50):
    idx = rng_w.integers(0, len(Xw_train), len(Xw_train))
    m = Pipeline([('scale', StandardScaler()),
                  ('model', LogisticRegression(l1_ratio=1, solver='liblinear', C=best_C))]).fit(Xw_train.iloc[idx], yw_train.iloc[idx])
    cnt_w += (m.named_steps['model'].coef_[0] != 0)
stab_w = (cnt_w / 50).sort_values(ascending=False)
stab_w.round(2)

### Step 8 · Summary table, final pipeline and test evaluation

In [ ]:
summary_w = pd.DataFrame({
    'F_rank': filt['F_rank'],
    'MI_rank': filt['MI_rank'],
    'L1_coef': coef_w.round(3),
    'RFECV_kept': rf_step.support_.astype(int),
    'perm_rank': perm_w['mean'].rank(ascending=False).astype(int),
    'L1_stability': stab_w,
}).sort_values('F_rank')
summary_w

In [ ]:
for k in [3, 4, 5, 6, 8, 11]:
    p = Pipeline([('scale', StandardScaler()), ('select', SelectKBest(f_classif, k=k)),
                  ('model', LogisticRegression(max_iter=5000))])
    print(f'SelectKBest k={k:>2}: CV ROC-AUC', auc(p, Xw_train))

In [ ]:
final_w = Pipeline([('scale', StandardScaler()),
                    ('select', SelectFromModel(LogisticRegression(l1_ratio=1, solver='liblinear', C=best_C))),
                    ('model', LogisticRegression(max_iter=5000))])
print('Final pipeline CV ROC-AUC:', auc(final_w, Xw_train), '| baseline:', auc(lr_w, Xw_train))
final_w.fit(Xw_train, yw_train)
kept_w = list(Xw_train.columns[final_w.named_steps['select'].get_support()])
print(f'{len(kept_w)} features kept:', kept_w)

from sklearn.metrics import roc_auc_score
print('\nTest ROC-AUC (touched once): final =', round(roc_auc_score(yw_test, final_w.predict_proba(Xw_test)[:, 1]), 4),
      '| baseline =', round(roc_auc_score(yw_test, lr_w.fit(Xw_train, yw_train).predict_proba(Xw_test)[:, 1]), 4))

### Conclusions — write your own from the tables above, then compare

- **Consistent winners:** `alcohol`, `volatile acidity` (negative) and `sulphates` rank at the top with almost every method
  and are selected in (nearly) every bootstrap — the same three features the Topic 7 EDA pointed to.
- **Borderline features** (e.g. `citric acid`, `total sulfur dioxide`, `density`, `fixed acidity`, `chlorides`) change rank between
  methods and appear in only some bootstraps: they add a little, and are partly redundant with each other.
- **Little evidence:** `pH`, `free sulfur dioxide` and `residual sugar` score low in the univariate tests.
- **`citric acid`** is 3rd in the univariate tests but L1 drops it (and keeps it in under half of the bootstraps): its
  information overlaps with `volatile acidity` and `fixed acidity` (correlated, see Topic 7). A univariate winner is not
  necessarily needed once other features are in the model.
- The final L1 pipeline keeps **7 of 11** features with practically the same CV ROC-AUC as all 11. On the test set it is
  about 0.006 lower — far inside the fold-to-fold std (≈0.04), so the honest conclusion is *"same quality, fewer inputs"*.
- Everything that looked at the target (tests, L1, RFECV, permutation) was done on the **training** split or inside CV;
  the test set was used once.

---
<a id="sec-21"></a>
# 📋 Feature Selection Cheat Sheet

## Families
| Family | Tools | Speed | Multivariate |
|---|---|---|---|
| Filter | `VarianceThreshold`, correlation, `SelectKBest/SelectPercentile` + `f_classif` · `chi2` · `mutual_info_*` · `f_regression` | ⚡ | ❌ |
| Wrapper | `RFE`, `RFECV`, `SequentialFeatureSelector` | 🐢 | ✅ |
| Embedded | L1 (`LogisticRegression(l1_ratio=1, solver='liblinear')`, `Lasso`, `LassoCV`) + `SelectFromModel`; tree `feature_importances_` | ⚡/🙂 | ✅ |
| Model-agnostic | `permutation_importance` on held-out data | 🙂 | ✅ |

## Junk removal
```python
VarianceThreshold(0)                                          # constants
X.apply(lambda s: s.value_counts(normalize=True).iloc[0])     # quasi-constant (scale-free)
X.columns[X.T.duplicated()]                                   # duplicate columns
corr.where(np.triu(np.ones(corr.shape, bool), 1))             # correlated pairs (|r| > 0.9–0.95)
```

## Score functions
| Target \ feature | numeric | categorical (one-hot) |
|---|---|---|
| classification | `f_classif`, `mutual_info_classif` | `chi2` (non-negative!), `mutual_info_classif(discrete_features=True)` |
| regression | `f_regression`, `mutual_info_regression` | `f_regression`, `mutual_info_regression(discrete_features=True)` |

## Pipelines (no leakage)
```python
pipe = Pipeline([('scale', StandardScaler()),
                 ('select', SelectKBest(f_classif)),
                 ('model', LogisticRegression(max_iter=5000))])
GridSearchCV(pipe, {'select__k': [5, 10, 20]}, cv=5).fit(X_train, y_train)

Pipeline([('scale', StandardScaler()),
          ('select', SelectFromModel(LogisticRegression(l1_ratio=1, solver='liblinear', C=0.1))),
          ('model', ...)])
RFECV(estimator, cv=5, scoring=...)            # chooses the number of features
SequentialFeatureSelector(est, n_features_to_select=5, direction='forward', cv=5)
permutation_importance(model, X_val, y_val, n_repeats=20, scoring=...)
```

## Rules
✔ split first; selection only on training data / inside CV ·
✔ scale before L1 / RFE with linear models · ✔ `chi2` needs non-negative input ·
✔ variance thresholds are scale-dependent · ✔ tree impurity importance favours noise — prefer permutation ·
✔ check stability with bootstraps · ✔ de-correlate or group correlated features ·
✔ prefer the simplest model within one std of the best · ✔ the test set is touched once

🎉 **You have completed Topic 16 — Feature Selection!** Next: **Topic 17 — Machine Learning**.